In [1]:
import os
from pathlib import Path

import sqlglot
from sqlglot import exp, expressions

from src.utils.file_utils import parse_file_name
from src.migration.decomposer import SqlDecomposer, DecomposerWriter
from src.migration.metadata import MetadataProcessor
from src.migration.generator import PySparkGenerator
from src.paths import *

In [2]:
USERNAME

'ext_giadung'

In [3]:
from src.utils.source_rule_loader import load_all_source_rules

# input_file = PROJECT_ROOT / "docs" / "datalake_old" /"dml" / "com_r_k2_cif_alias.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_r_mhbos_m_client_crs.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_t_mhbos_m_client.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_contact.sql"
input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer.sql"
file_name = os.path.basename(input_file).replace('.sql', '')
layer, sub_layer, source_name, base_table = parse_file_name(input_file)
output_root = PROJECT_ROOT / "output" / "migration"

all_source_rules = load_all_source_rules()
source_rules = load_all_source_rules()[source_name] if source_name in all_source_rules else all_source_rules['default']

In [4]:
# def run_migration_pipeline():
# ==========================================
# BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
# ==========================================
decomposer = SqlDecomposer(source_rules)
decomposed_script = decomposer.decompose(input_file)

# Ghi file sub-SQL ra ổ đĩa
writer = DecomposerWriter()
writer.write(decomposed_script, output_root / file_name)
print(f"✅ [Bước 1] Đã bóc tách thành các block tại: {output_root / file_name / 'processing_steps'}")

# ==========================================
# BƯỚC 2: XỬ LÝ METADATA (PROCESSOR)
# ==========================================
processor = MetadataProcessor(source_rules, ai_fallback=False)

try:
    # Hàm này sẽ phân tích AST, tự động tìm file DDL và trích xuất Schema
    pipeline_config = processor.process(decomposed_script, input_file)

    # Ghi file YAML
    metadata_output_dir = output_root / file_name / "metadata"
    processor.write_yaml(pipeline_config, metadata_output_dir)

    print(f"✅ [Bước 2] Đã xử lý Metadata thành công!")
    print(f"   -> Model nhận diện được: Model {pipeline_config['model_type']}")
    print(f"   -> Khóa (Key) nhận diện được: {pipeline_config['key']}")
    print(f"   -> File YAML đã lưu tại: {metadata_output_dir / (file_name + '.yaml')}")

except FileNotFoundError as e:
    print(f"❌ [Lỗi Bước 2]: {e}")
    print("💡 Gợi ý: Hãy đảm bảo bạn có file DDL tương ứng tại `docs/datalake_old/ddl/com_r_k2_cif_alias.sql` hoặc cùng thư mục `dml/` để hàm trích xuất Schema hoạt động!")


✅ [Bước 1] Đã bóc tách thành các block tại: C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_dim_customer\processing_steps
✅ [Bước 2] Đã xử lý Metadata thành công!
   -> Model nhận diện được: Model 3a
   -> Khóa (Key) nhận diện được: customer_id
   -> File YAML đã lưu tại: C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_dim_customer\metadata\cur_dim_customer.yaml


In [5]:
from migration.metadata import SchemaExtractor


SchemaExtractor(source_rules).extract(Path(r"C:\Users\ext_giadung\projects\datalake-script\ddl\com\com_m_mhbos_m_client.sql"))


[{'name': 'cust_id', 'type': 'VARCHAR(11)', 'remark': None},
 {'name': 'client_no', 'type': 'VARCHAR(9)', 'remark': None},
 {'name': 'clean_rule_flag', 'type': 'VARCHAR(60)', 'remark': None},
 {'name': 'primary_identification_type',
  'type': 'VARCHAR(10)',
  'remark': None},
 {'name': 'primary_identification_no', 'type': 'VARCHAR(60)', 'remark': None},
 {'name': 'secondary_identification_type',
  'type': 'VARCHAR(10)',
  'remark': None},
 {'name': 'secondary_identification_no',
  'type': 'VARCHAR(60)',
  'remark': None},
 {'name': 'customer_name', 'type': 'VARCHAR(250)', 'remark': None},
 {'name': 'customer_name_concatenate', 'type': 'VARCHAR(250)', 'remark': None},
 {'name': 'client_name', 'type': 'VARCHAR(60)', 'remark': None},
 {'name': 'client_name1', 'type': 'VARCHAR(60)', 'remark': None},
 {'name': 'client_name2', 'type': 'VARCHAR(60)', 'remark': None},
 {'name': 'client_name3', 'type': 'VARCHAR(60)', 'remark': None},
 {'name': 'mobile_no', 'type': 'VARCHAR(20)', 'remark': None}

In [6]:
print("==========================================")
print(" BƯỚC 3: SINH CODE PYSPARK (GENERATOR)")
print("==========================================")

with open(output_root / file_name / "metadata" / (file_name + '.yaml'), 'r') as f:
    pipeline_config = yaml.safe_load(f)

generator = PySparkGenerator(source_rules, output_mode="simple")
ddl_context, dml_context = generator.generate(pipeline_config, output_root / file_name)

print("🎉 Hoàn tất toàn bộ Pipeline!")

 BƯỚC 3: SINH CODE PYSPARK (GENERATOR)
Generated DDL at C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_dim_customer\ddl\cur_None_dim_customer.sql
Reading pre-processing SQL from C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_dim_customer\processing_steps\cur_TEMP_DIM_CUSTOMER_MHBOS.sql
TEMP None
Reading pre-processing SQL from C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_dim_customer\processing_steps\cur_TEMP_DIM_CUSTOMER_GUAVA_COMPANY.sql
TEMP None
Reading pre-processing SQL from C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_dim_customer\processing_steps\cur_TEMP_DIM_CUSTOMER_GUAVA_CUSTOMER.sql
TEMP None
Reading pre-processing SQL from C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_dim_customer\processing_steps\cur_TEMP_DIM_CUSTOMER_M21.sql
TEMP None
Reading pre-processing SQL from C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_dim_customer\processing_steps\

In [7]:
pipeline_config['pre_processing']

[{'name': 'TEMP_DIM_CUSTOMER_MHBOS',
  'file': 'output/migration/cur_dim_customer/processing_steps/cur_TEMP_DIM_CUSTOMER_MHBOS.sql',
  'action': 'include',
  'dependencies': [{'TEMP_DIM_CUSTOMER_MHBOS': {'schema': 'cur'}},
   {'M_MHBOS_M_CLIENT': {'schema': 'com'}},
   {'T_MHBOS_M_CLIENT_EXT': {'schema': 'com'}},
   {'DIM_DECLARATION': {'schema': 'cur'}},
   {'DIM_FATCA': {'schema': 'cur'}},
   {'R_MHBOS_M_CLIENT_CRS': {'schema': 'com'}},
   {'R_K2_ACCOUNT': {'schema': 'com'}},
   {'R_K2_CIF_ACCOUNT': {'schema': 'com'}},
   {'R_K2_CIF_EXT': {'schema': 'com'}},
   {'REF_COUNTRY': {'schema': 'cur'}},
   {'R_K2_CIF_ALIAS': {'schema': 'com'}},
   {'R_K2_RULE_VALUE': {'schema': 'com'}}]},
 {'name': 'TEMP_DIM_CUSTOMER_GUAVA_COMPANY',
  'file': 'output/migration/cur_dim_customer/processing_steps/cur_TEMP_DIM_CUSTOMER_GUAVA_COMPANY.sql',
  'action': 'include',
  'dependencies': [{'TEMP_DIM_CUSTOMER_GUAVA_COMPANY': {'schema': 'cur'}},
   {'M_GUAVA_COMPANY': {'schema': 'com'}},
   {'T_GUAVA_BRAN

In [8]:
dml_context['pre_processing_sqls']

['/* ==============[Group.1]============== */\nDROP TABLE IF EXISTS {params["cur_schema"]}.TEMP_DIM_CUSTOMER_MHBOS',
 'CREATE TABLE {params["cur_schema"]}.TEMP_DIM_CUSTOMER_MHBOS (\n  CUSTOMER_ID VARCHAR(20), /* None */\n  CUSTOMER_TYPE VARCHAR(10), /* None */\n  CUSTOMER_TITLE VARCHAR(20), /* None */\n  CUSTOMER_NAME VARCHAR(250), /* None */\n  CUSTOMER_PRIMARY_IDENTIFICATION_NO_TYPE VARCHAR(20), /* None */\n  CUSTOMER_PRIMARY_IDENTIFICATION_NO VARCHAR(20), /* None */\n  CUSTOMER_PRIMARY_IDENTIFICATION_NO_EXPIRY_DATE DATE, /* None */\n  CUSTOMER_SECONDARY_IDENTIFICATION_NO_TYPE VARCHAR(20), /* None */\n  CUSTOMER_SECONDARY_IDENTIFICATION_NO VARCHAR(20), /* None */\n  CUSTOMER_SECONDARY_IDENTIFICATION_NO_EXPIRY_DATE DATE, /* None */\n  CUSTOMER_NATIONALITY VARCHAR(2), /* None */\n  CUSTOMER_COUNTRY_OF_RESIDENCE VARCHAR(2), /* None */\n  CUSTOMER_COUNTRY_OF_BIRTH VARCHAR(2), /* None */\n  CUSTOMER_DATE_OF_BIRTH DATE, /* None */\n  CUSTOMER_BUMIPUTRA_STATUS VARCHAR(1), /* None */\n  CUST

In [9]:

node = decomposed_script.temp_tables[0].ast_nodes[3].copy()


for table_node in node.find_all(exp.Table):
    print(table_node)
    if table_node.name.startswith("r_"):
        new_name = table_node.name[2:]
        table_node.set("this", exp.Identifier(this=new_name, quoted=table_node.this.args.get("quoted", False)))
        table_node.set("db", exp.Identifier(this=exp.Parameter(this=exp.Var(this="batch_date"))))
        print("Renamed to: ", table_node)


@cur_schema.TEMP_DIM_CUSTOMER_MHBOS


In [10]:

# Read pre_processing SQLs
pre_processing_sqls = []
for step in pipeline_config.get("pre_processing", []):
    if step.get("action") == "skip":
        continue
    step_file = Path(step["file"])
    if step_file.exists():
        pre_processing_sqls.append(step_file.read_text(encoding="utf-8"))